# 03 — MSRS Temporal Transition Discovery

Apply Maximally Selected Rank Statistics (MSRS) to discover data-driven temporal
cut-points between early and late recurrence phases. The primary discovery cohorts
are GSE2034 (breast, 107 events, 86mo follow-up) and GSE31210 (LUAD, 64 events,
54mo follow-up). TCGA-BRCA provides molecular validation.

Three steps:
1. MSRS scan with permutation significance test
2. Phase comparison: module scores in early vs late recurrence events
3. JSD divergence per pathway at the discovered cut-point
4. Cancer-type-specific axes for GBM (invasion-hypoxia) and KIRC (angiogenesis)


In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import mannwhitneyu

warnings.filterwarnings("ignore")
np.random.seed(42)

BASE_DIR = Path(os.environ.get("COSMOS_BASE_DIR", "./data"))
RAW_DIR = BASE_DIR / "Raw Data"
INTER_DIR = BASE_DIR / "intermediates"
DAYS_PER_MO = 365.25 / 12

MODULES = ["Proliferation", "Immune", "Metabolic", "Microenvironment"]


## Load intermediates


In [ ]:
def load_pkl(name):
    with open(INTER_DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)


def save_pkl(obj, name):
    with open(INTER_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


module_scores_37 = load_pkl("module_scores_37")
ssgsea_corrected = load_pkl("ssgsea_corrected")
geo_clinical = load_pkl("geo_clinical")
tcga_clinical = load_pkl("tcga_clinical")
PATHWAY_MODULES_37 = load_pkl("core_pathways_37")
BRCA_SPECIFIC_MODULES = load_pkl("brca_specific_modules")
LUAD_SPECIFIC_MODULES = load_pkl("luad_specific_modules")

ALL_37 = [p for ps in PATHWAY_MODULES_37.values() for p in ps]
PW_TO_MOD = {p: m for m, ps in PATHWAY_MODULES_37.items() for p in ps}

cdr_path = RAW_DIR / "tcga_clinical" / "TCGA-CDR.csv"
cdr = pd.read_csv(cdr_path) if cdr_path.exists() else pd.read_excel(cdr_path.with_suffix(".xlsx"))


def cdr_endpoint(cancer, ep):
    """Extract endpoint time/event from CDR for a given cancer type."""
    sub = cdr[cdr["type"] == cancer].copy()
    if "bcr_patient_barcode" in sub.columns:
        sub = sub.set_index("bcr_patient_barcode")
    tc = f"{ep}.time"
    if tc not in sub.columns:
        return None
    out = pd.DataFrame({
        "time": pd.to_numeric(sub[tc], errors="coerce") / DAYS_PER_MO,
        "event": (pd.to_numeric(sub[ep], errors="coerce") > 0).astype(float),
    }, index=sub.index).dropna()
    out.index = out.index.astype(str).str[:12]
    return out[out["time"] > 0].loc[lambda x: ~x.index.duplicated()]


print("Intermediates loaded")


## Build cohort DataFrames


In [ ]:
def to_array(series):
    return pd.to_numeric(series, errors="coerce").dropna().values.astype(np.float64)


def cohen_d(a, b):
    return (a.mean() - b.mean()) / (np.sqrt((a.std() ** 2 + b.std() ** 2) / 2) + 1e-10)


def build_cohort_df(key, clin_override=None):
    """Build a per-cohort dataframe of sample, time, event, and module scores."""
    ms = module_scores_37.get(key)
    if ms is None:
        return None
    if clin_override is not None:
        clin = clin_override
    elif geo_clinical.get(key) is not None:
        clin = geo_clinical.get(key)
    else:
        clin = tcga_clinical.get(key)
    if clin is None or clin.empty:
        return None
    ms_T = ms.T
    common = sorted(set(ms_T.index.astype(str)) & set(clin.index.astype(str)))
    if len(common) < 20:
        return None
    rows = []
    for s in common:
        r = {
            "sample": s, "cohort": key,
            "time": float(clin.loc[s, "time"]),
            "event": float(clin.loc[s, "event"]),
        }
        for m in MODULES:
            r[m] = float(pd.to_numeric(
                ms_T.loc[s, m] if m in ms_T.columns else np.nan, errors="coerce"
            ))
        rows.append(r)
    df = pd.DataFrame(rows).dropna(subset=["time", "event", "Proliferation"])
    return df[(df["time"] > 0) & df["event"].isin([0., 1.])].reset_index(drop=True)


BREAST_GEO = ["GSE2034", "GSE2990", "GSE103746"]
breast_df = pd.concat(
    [build_cohort_df(c) for c in BREAST_GEO if build_cohort_df(c) is not None],
    ignore_index=True,
)
gse2034_df = build_cohort_df("GSE2034")
gse31210_df = build_cohort_df("GSE31210")
brca_df = build_cohort_df("TCGA-BRCA", cdr_endpoint("BRCA", "DFI"))
luad_pfi_df = build_cohort_df("TCGA-LUAD")
gbm_df = build_cohort_df("TCGA-GBM")
kirc_df = build_cohort_df("TCGA-KIRC")
kirc_os_df = build_cohort_df("TCGA-KIRC", cdr_endpoint("KIRC", "OS"))

for name, df in [
    ("Breast GEO pooled", breast_df),
    ("GSE2034", gse2034_df),
    ("GSE31210", gse31210_df),
    ("TCGA-BRCA (DFI)", brca_df),
    ("TCGA-LUAD (PFI)", luad_pfi_df),
    ("TCGA-GBM", gbm_df),
    ("TCGA-KIRC (PFI)", kirc_df),
    ("TCGA-KIRC (OS)", kirc_os_df),
]:
    if df is not None:
        print(f"  {name}: n={len(df)}, events={int(df['event'].sum())}, "
              f"median FU={df['time'].median():.1f}mo")


## MSRS scan

Scan all candidate cut-points between 6 and 60 months. At each candidate, compute
the Mann-Whitney U statistic between recurrence events on either side using the
specified score column. The optimal cut-point maximises U; significance is
assessed by permutation test on the score vector.


In [ ]:
def msrs(df, label, score_col="Proliferation",
         t_min=6, t_max=60, min_each=20, n_perm=10000, seed=42):
    """Maximally Selected Rank Statistics. Scans cut-points in [t_min, t_max]
    and reports the U-maximizing cut with a permutation p-value."""
    if df is None or df.empty:
        return None, None, None
    rng = np.random.RandomState(seed)
    ev = df[df["event"] == 1].copy()
    t = ev["time"].values.astype(float)
    s = to_array(ev[score_col])
    n = min(len(t), len(s))
    t, s = t[:n], s[:n]

    rows = []
    for cut in range(t_min, t_max + 1):
        em = t <= cut
        lm = t > cut
        if em.sum() < min_each or lm.sum() < min_each:
            continue
        u, _ = mannwhitneyu(s[em], s[lm], alternative="greater")
        rows.append({
            "timepoint": cut, "u_stat": float(u),
            "cohen_d": float(cohen_d(s[em], s[lm])),
            "n_early": int(em.sum()), "n_late": int(lm.sum()),
        })
    if not rows:
        return None, None, None

    scan = pd.DataFrame(rows).set_index("timepoint")
    opt = int(scan["u_stat"].idxmax())
    umax = float(scan["u_stat"].max())
    cands = scan.index.tolist()

    perm_max = []
    for _ in range(n_perm):
        sp = rng.permutation(s)
        pm = [
            mannwhitneyu(sp[t <= c], sp[t > c], alternative="greater")[0]
            for c in cands
            if (t <= c).sum() >= min_each and (t > c).sum() >= min_each
        ]
        if pm:
            perm_max.append(max(pm))
    perm_p = (np.sum(np.array(perm_max) >= umax) + 1) / (len(perm_max) + 1)
    r = scan.loc[opt]
    print(f"  {label}: opt={opt}mo, d={r['cohen_d']:.3f}, "
          f"early/late={int(r['n_early'])}/{int(r['n_late'])}, P={perm_p:.4f}")
    return scan, opt, perm_p


In [ ]:
print("MSRS discovery scans")
scan_gse2034, opt_gse2034, pp_gse2034 = msrs(
    gse2034_df, "GSE2034 (breast)", min_each=15, n_perm=10000,
)
scan_gse, opt_gse, pp_gse = msrs(
    gse31210_df, "GSE31210 (LUAD)",
    t_min=8, t_max=50, min_each=8, n_perm=5000,
)
scan_brca, opt_brca, pp_brca = msrs(
    brca_df, "TCGA-BRCA (DFI)",
    t_min=12, t_max=42, min_each=8, n_perm=5000,
)

T_BREAST = opt_gse2034 if opt_gse2034 else 25
T_LUAD = opt_gse if opt_gse else 21
print(f"\nCanonical cut-points: breast={T_BREAST}m, LUAD={T_LUAD}m")


## JSD divergence at the discovered cut-point

For each of the 37 pre-specified pathways, compute the Jensen-Shannon divergence
between early and late distributions at the MSRS-discovered cut-point. This is
a non-parametric test of whether the two distributions are genuinely different,
independent of mean separation.


In [ ]:
def jsd_at_split(ss_list, clin_list, cohort_names, t, label, n_bins=20):
    """JSD per pathway pooled across cohorts at a fixed cut-point."""
    rows = []
    for cohort, ss, clin in zip(cohort_names, ss_list, clin_list):
        if ss is None or clin is None:
            continue
        ss_T = ss.T
        common = sorted(set(ss_T.index.astype(str)) & set(clin.index.astype(str)))
        ss_s = ss_T.loc[common]
        cl_s = clin.loc[common]
        ev = cl_s[cl_s["event"] == 1]
        ei = ev.index[ev["time"] <= t].tolist()
        li = ev.index[ev["time"] > t].tolist()
        if len(ei) < 5 or len(li) < 5:
            continue
        for pw in ALL_37:
            if pw not in ss_s.columns:
                continue
            e = to_array(ss_s.loc[ei, pw])
            l = to_array(ss_s.loc[li, pw])
            if len(e) < 3 or len(l) < 3:
                continue
            lo = np.percentile(np.concatenate([e, l]), 2)
            hi = np.percentile(np.concatenate([e, l]), 98)
            if hi <= lo:
                continue
            bins = np.linspace(lo, hi, n_bins + 1)
            pe, _ = np.histogram(e, bins=bins)
            pl, _ = np.histogram(l, bins=bins)
            pe = (pe + 0.5) / (pe + 0.5).sum()
            pl = (pl + 0.5) / (pl + 0.5).sum()
            rows.append({
                "cohort": cohort, "pathway": pw,
                "module": PW_TO_MOD[pw],
                "jsd": float(jensenshannon(pe, pl, base=2)),
                "n_early": len(e), "n_late": len(l),
            })
    if not rows:
        return None, None
    jsd_df = pd.DataFrame(rows)
    agg = jsd_df.groupby("pathway").agg(
        module=("module", "first"),
        mean_jsd=("jsd", "mean"),
        n_cohorts=("cohort", "nunique"),
    ).reset_index().sort_values("mean_jsd", ascending=False)
    print(f"  {label}: mean JSD={agg['mean_jsd'].mean():.3f}, "
          f"high-divergence pathways (>0.3)={int((agg['mean_jsd']>0.3).sum())}/{len(agg)}")
    return agg, jsd_df


breast_ss = [ssgsea_corrected.get(c) for c in BREAST_GEO]
breast_clin = [geo_clinical.get(c) for c in BREAST_GEO]
breast_jsd_agg, breast_jsd_df = jsd_at_split(
    breast_ss, breast_clin, BREAST_GEO, T_BREAST, "Breast GEO pooled",
)
luad_jsd_agg, luad_jsd_df = jsd_at_split(
    [ssgsea_corrected.get("GSE31210")], [geo_clinical.get("GSE31210")],
    ["GSE31210"], T_LUAD, "GSE31210 LUAD",
)
brca_jsd_agg, brca_jsd_df = jsd_at_split(
    [ssgsea_corrected.get("TCGA-BRCA")], [cdr_endpoint("BRCA", "DFI")],
    ["TCGA-BRCA"], T_BREAST, "TCGA-BRCA DFI validation",
)


## Phase comparison

For each cohort and biological module, compare early vs late recurrence event
distributions at the cohort-appropriate cut-point. Proliferation is expected to
be elevated in early-phase recurrence; immune and microenvironment modules are
expected to be elevated in late-phase recurrence.


In [ ]:
def significance(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def phase_compare(df, t, label, min_each=5):
    """Module-score comparison between early and late recurrence events."""
    if df is None:
        return {}, pd.DataFrame(), pd.DataFrame()
    ev = df[df["event"] == 1].copy()
    early = ev[ev["time"] <= t]
    late = ev[ev["time"] > t]
    print(f"  {label} (split {t}m): early n={len(early)}, late n={len(late)}")
    if len(early) < min_each or len(late) < min_each:
        return {}, early, late
    res = {}
    for mod in MODULES:
        if mod not in ev.columns:
            continue
        e = to_array(early[mod])
        l = to_array(late[mod])
        if len(e) < 3 or len(l) < 3:
            continue
        _, p = mannwhitneyu(e, l, alternative="two-sided")
        d = cohen_d(e, l)
        expected = (mod == "Proliferation" and d > 0) or (mod != "Proliferation" and d < 0)
        mark = "ok" if expected else "X"
        print(f"    [{mark}] {mod}: d={d:+.3f}, P={p:.2e} {significance(p)}")
        res[mod] = {
            "cohen_d": float(d), "p_val": float(p), "correct": expected,
            "n_early": len(e), "n_late": len(l),
            "mean_early": float(e.mean()), "std_early": float(e.std()),
            "mean_late": float(l.mean()), "std_late": float(l.std()),
        }
    return res, early, late


print("Discovery phase comparisons")
breast_phase, breast_early, breast_late = phase_compare(
    breast_df, T_BREAST, "Pooled Breast GEO",
)
gse_phase, gse_early, gse_late = phase_compare(
    gse31210_df, T_LUAD, "GSE31210 LUAD",
)

print("\nTCGA validation phase comparisons")
brca_phase, _, _ = phase_compare(brca_df, T_BREAST, "TCGA-BRCA (DFI)")
luad_pfi_phase, _, _ = phase_compare(luad_pfi_df, T_LUAD, "TCGA-LUAD (PFI)")


## Pathway-level evidence

Direction-of-effect (late minus early) for each of the 37 pathways pooled across
discovery cohorts.


In [ ]:
def pathway_evidence(cohort_list, clin_dict, ss_dict, t, label):
    """Pathway-level early vs late comparison pooled across cohorts."""
    rows = []
    for cohort in cohort_list:
        ss = ss_dict.get(cohort)
        clin = clin_dict.get(cohort)
        if ss is None or clin is None or clin.empty:
            continue
        ss_T = ss.T
        common = sorted(set(ss_T.index.astype(str)) & set(clin.index.astype(str)))
        ss_s = ss_T.loc[common]
        cl_s = clin.loc[common]
        ev = cl_s[cl_s["event"] == 1]
        ei = ev.index[ev["time"] <= t].tolist()
        li = ev.index[ev["time"] > t].tolist()
        for pw in ALL_37:
            if pw not in ss_s.columns:
                continue
            e = to_array(ss_s.loc[ei, pw]) if ei else np.array([], dtype=float)
            l = to_array(ss_s.loc[li, pw]) if li else np.array([], dtype=float)
            if len(e) < 3 or len(l) < 3:
                continue
            rows.append({
                "cohort": cohort, "pathway": pw,
                "module": PW_TO_MOD[pw],
                "diff": float(l.mean() - e.mean()),
                "mean_early": float(e.mean()),
                "mean_late": float(l.mean()),
            })
    if not rows:
        return pd.DataFrame()
    pw = pd.DataFrame(rows)
    agg = pw.groupby("pathway").agg(
        module=("module", "first"),
        diff=("diff", "mean"),
        mean_early=("mean_early", "mean"),
        mean_late=("mean_late", "mean"),
    ).reset_index().sort_values("diff")

    n_pr = int((pw[pw["module"] == "Proliferation"]["diff"] < 0).sum())
    n_im = int((pw[pw["module"] == "Immune"]["diff"] > 0).sum())
    n_me = int((pw[pw["module"] == "Microenvironment"]["diff"] > 0).sum())
    t_pr = len(pw[pw["module"] == "Proliferation"])
    t_im = len(pw[pw["module"] == "Immune"])
    t_me = len(pw[pw["module"] == "Microenvironment"])
    print(f"  {label} (split {t}m):")
    print(f"    Proliferation early > late: {n_pr}/{t_pr}")
    print(f"    Immune late > early:        {n_im}/{t_im}")
    print(f"    Microenvironment late>early:{n_me}/{t_me}")
    return agg


breast_pw_agg = pathway_evidence(
    BREAST_GEO, geo_clinical, ssgsea_corrected, T_BREAST, "Breast GEO",
)
luad_pw_agg = pathway_evidence(
    ["GSE31210"], geo_clinical, ssgsea_corrected, T_LUAD, "GSE31210 LUAD",
)


## Pathway-level FDR correction

For each of the 37 pre-specified pathways, pool early- and late-recurrence-event
NES scores across the four discovery cohorts (GSE2034, GSE2990, GSE103746,
GSE31210) and run a Mann-Whitney U test, then apply Benjamini-Hochberg correction
across all 37 pathways. This is the internal consistency check reported in
Section 3.2 of the manuscript ("36 of 37 pathways significant after BH").


In [ ]:
from statsmodels.stats.multitest import multipletests


def pathway_fdr_bh(cohort_list, clin_dict, ss_dict, cohort_cuts, label,
                   pathways=ALL_37, alpha=0.05):
    """Pool early/late NES across cohorts, run Mann-Whitney U per pathway,
    then apply Benjamini-Hochberg correction across pathways.

    cohort_cuts: dict {cohort_name: cut_in_months}
    """
    pooled_early = {pw: [] for pw in pathways}
    pooled_late = {pw: [] for pw in pathways}

    for cohort in cohort_list:
        ss = ss_dict.get(cohort)
        clin = clin_dict.get(cohort)
        t = cohort_cuts.get(cohort)
        if ss is None or clin is None or clin.empty or t is None:
            continue
        ss_T = ss.T
        common = sorted(set(ss_T.index.astype(str)) & set(clin.index.astype(str)))
        ss_s = ss_T.loc[common]
        cl_s = clin.loc[common]
        ev = cl_s[cl_s["event"] == 1]
        ei = ev.index[ev["time"] <= t].tolist()
        li = ev.index[ev["time"] > t].tolist()
        for pw in pathways:
            if pw not in ss_s.columns:
                continue
            e = to_array(ss_s.loc[ei, pw]) if ei else np.array([], dtype=float)
            l = to_array(ss_s.loc[li, pw]) if li else np.array([], dtype=float)
            if len(e) >= 3:
                pooled_early[pw].extend(e)
            if len(l) >= 3:
                pooled_late[pw].extend(l)

    rows = []
    raw_pvals = []
    pathway_order = []
    for pw in pathways:
        e = np.asarray(pooled_early[pw], float)
        l = np.asarray(pooled_late[pw], float)
        if len(e) < 5 or len(l) < 5:
            continue
        try:
            _, p = mannwhitneyu(e, l, alternative="two-sided")
        except ValueError:
            continue
        d = cohen_d(e, l)
        rows.append({
            "pathway": pw,
            "module": PW_TO_MOD[pw],
            "cohen_d": float(d),
            "p_raw": float(p),
            "n_early": int(len(e)),
            "n_late": int(len(l)),
            "mean_early": float(e.mean()),
            "mean_late": float(l.mean()),
        })
        raw_pvals.append(float(p))
        pathway_order.append(pw)

    if not rows:
        print(f"  {label}: no pathways tested")
        return None

    _, p_bh, _, _ = multipletests(raw_pvals, method="fdr_bh")
    fdr_map = dict(zip(pathway_order, p_bh))
    for r in rows:
        r["p_bh"] = float(fdr_map[r["pathway"]])

    df = pd.DataFrame(rows).sort_values("p_bh")
    n_sig = int((df["p_bh"] < alpha).sum())
    print(f"  {label}: {n_sig}/{len(df)} pathways significant at FDR < {alpha}")
    print(f"    Per-module FDR<{alpha} counts:")
    for mod in MODULES:
        sub = df[df["module"] == mod]
        n_sig_mod = int((sub["p_bh"] < alpha).sum())
        print(f"      {mod}: {n_sig_mod}/{len(sub)}")
    return df


BREAST_GEO_CUTS = {c: T_BREAST for c in BREAST_GEO}
breast_fdr_df = pathway_fdr_bh(
    BREAST_GEO, geo_clinical, ssgsea_corrected, BREAST_GEO_CUTS,
    "Breast GEO pooled",
)

# Combined pooled analysis across discovery cohorts (breast at T_BREAST, LUAD at T_LUAD)
DISCOVERY_CUTS = {**{c: T_BREAST for c in BREAST_GEO}, "GSE31210": T_LUAD}
discovery_fdr_df = pathway_fdr_bh(
    BREAST_GEO + ["GSE31210"], geo_clinical, ssgsea_corrected, DISCOVERY_CUTS,
    "Discovery pooled (breast at {}m, LUAD at {}m)".format(T_BREAST, T_LUAD),
)


## Bootstrap confidence intervals for Cohen's d

For the pooled-breast discovery cohort and the smaller individual cohorts
(GSE2990, GSE103746), compute 95% bootstrap CIs around the Cohen's d of the
Proliferation module. Bootstrap is 10,000 resamples with replacement.


In [ ]:
def bootstrap_cohen_d_ci(early, late, n_boot=10000, seed=42, alpha=0.05):
    """Compute percentile bootstrap CI for Cohen's d between two arrays."""
    rng = np.random.RandomState(seed)
    early = np.asarray(early, float)
    late = np.asarray(late, float)
    if len(early) < 2 or len(late) < 2:
        return None, None, None
    ds = np.empty(n_boot)
    for i in range(n_boot):
        eb = early[rng.randint(0, len(early), len(early))]
        lb = late[rng.randint(0, len(late), len(late))]
        ds[i] = cohen_d(eb, lb)
    point = cohen_d(early, late)
    lo = float(np.percentile(ds, 100 * alpha / 2))
    hi = float(np.percentile(ds, 100 * (1 - alpha / 2)))
    return float(point), lo, hi


def proliferation_bootstrap(df, t, label):
    """Report Cohen's d with bootstrap CI for the Proliferation module."""
    if df is None or "Proliferation" not in df.columns:
        return None
    ev = df[df["event"] == 1]
    early = to_array(ev[ev["time"] <= t]["Proliferation"])
    late = to_array(ev[ev["time"] > t]["Proliferation"])
    d, lo, hi = bootstrap_cohen_d_ci(early, late)
    if d is None:
        return None
    print(f"  {label}: d = {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}], "
          f"early n = {len(early)}, late n = {len(late)}")
    return {"cohen_d": d, "ci_lo": lo, "ci_hi": hi,
            "n_early": len(early), "n_late": len(late)}


print("Bootstrap 95% CI for Proliferation Cohen's d (10,000 resamples):")
gse2990_df = build_cohort_df("GSE2990")
gse103746_df = build_cohort_df("GSE103746")
bs_results = {
    "Pooled breast GEO": proliferation_bootstrap(breast_df, T_BREAST, "Pooled breast GEO"),
    "GSE2034": proliferation_bootstrap(gse2034_df, T_BREAST, "GSE2034"),
    "GSE2990": proliferation_bootstrap(gse2990_df, T_BREAST, "GSE2990"),
    "GSE103746": proliferation_bootstrap(gse103746_df, T_BREAST, "GSE103746"),
    "GSE31210 LUAD": proliferation_bootstrap(gse31210_df, T_LUAD, "GSE31210 LUAD"),
}


## Cancer-type-specific biological axes

GBM and KIRC are not driven by a proliferation-to-immune transition; they have
distinct primary biology. We test cancer-specific composite scores instead.

- **GBM**: invasion + hypoxia composite (EGFR-driven proliferation-to-invasion switch)
- **KIRC**: angiogenesis composite (VHL-driven HIF pathway), tested with both PFI
  and OS endpoints because KIRC follow-up is too short for PFI alone.


In [ ]:
def build_composite(tcga_key, pathways, label):
    """Build a composite score from mean NES across specified pathways."""
    ss = ssgsea_corrected.get(tcga_key)
    clin = tcga_clinical.get(tcga_key)
    if ss is None or clin is None:
        return None
    ss_T = ss.T
    common = sorted(set(ss_T.index.astype(str)) & set(clin.index.astype(str)))
    ss_s = ss_T.loc[common]
    cl_s = clin.loc[common]
    avail = [p for p in pathways if p in ss_s.columns]
    if not avail:
        return None
    composite = ss_s[avail].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    df = cl_s[["time", "event"]].copy()
    df["composite_score"] = composite
    df = df.dropna().loc[lambda x: (x["time"] > 0) & x["event"].isin([0., 1.])]
    print(f"  {label}: n={len(df)}, events={int(df['event'].sum())}, "
          f"{len(avail)}/{len(pathways)} pathways available")
    return df.reset_index(drop=True)


GBM_INVASION_HX = [
    "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
    "HALLMARK_HYPOXIA",
    "KEGG_ECM_RECEPTOR_INTERACTION",
    "HALLMARK_TGF_BETA_SIGNALING",
    "HALLMARK_ANGIOGENESIS",
]

gbm_comp_df = build_composite("TCGA-GBM", GBM_INVASION_HX, "GBM invasion+hypoxia")
if gbm_comp_df is not None:
    scan_gbm, opt_gbm, pp_gbm = msrs(
        gbm_comp_df, "GBM (invasion+hypoxia)",
        score_col="composite_score",
        t_min=3, t_max=18, min_each=15, n_perm=10000,
    )
else:
    scan_gbm, opt_gbm, pp_gbm = None, 5, None


In [ ]:
# KIRC angiogenesis composite. Try VEGF in available namespaces.
kirc_ss = ssgsea_corrected.get("TCGA-KIRC")
kirc_keys = set(kirc_ss.index) if kirc_ss is not None else set()
vegf_path = None
for candidate in ["HALLMARK_VEGF_SIGNALING", "REACTOME_SIGNALING_BY_VEGF"]:
    if candidate in kirc_keys:
        vegf_path = candidate
        break

KIRC_ANGIO = [p for p in [
    "HALLMARK_ANGIOGENESIS",
    "HALLMARK_HYPOXIA",
    "HALLMARK_COAGULATION",
    vegf_path,
] if p is not None]
KIRC_ANGIO = list(dict.fromkeys(KIRC_ANGIO))
print(f"KIRC composite uses {len(KIRC_ANGIO)} pathways")

kirc_comp_df = build_composite("TCGA-KIRC", KIRC_ANGIO, "KIRC angiogenesis (PFI)")
kirc_comp_os_df = None
kirc_os_clin = cdr_endpoint("KIRC", "OS")
if kirc_os_clin is not None and kirc_ss is not None:
    avail_os = [p for p in KIRC_ANGIO if p in kirc_ss.index]
    common_os = sorted(
        set(kirc_ss.columns.astype(str)) & set(kirc_os_clin.index.astype(str))
    )
    if len(common_os) >= 20 and avail_os:
        comp_os = kirc_ss.loc[avail_os, common_os].apply(
            pd.to_numeric, errors="coerce"
        ).mean(0)
        kirc_comp_os_df = kirc_os_clin.loc[common_os].copy()
        kirc_comp_os_df["composite"] = comp_os
        kirc_comp_os_df = kirc_comp_os_df.dropna()

opt_kirc = pp_kirc = scan_kirc = None
opt_kirc_os = pp_kirc_os = scan_kirc_os = None
if kirc_comp_df is not None:
    scan_kirc, opt_kirc, pp_kirc = msrs(
        kirc_comp_df, "KIRC PFI",
        score_col="composite_score",
        t_min=12, t_max=60, min_each=20, n_perm=10000,
    )
if kirc_comp_os_df is not None and len(kirc_comp_os_df) >= 50:
    t_max_os = min(int(kirc_comp_os_df["time"].quantile(0.85)), 120)
    scan_kirc_os, opt_kirc_os, pp_kirc_os = msrs(
        kirc_comp_os_df, "KIRC OS",
        score_col="composite",
        t_min=12, t_max=t_max_os, min_each=20, n_perm=10000,
    )


## Cross-cancer summary


In [ ]:
summary_rows = [
    ("GBM", "invasion+hypoxia (own axis)", opt_gbm, pp_gbm),
    ("Breast", "proliferation MSRS", T_BREAST, pp_gse2034),
    ("LUAD", "proliferation MSRS", T_LUAD, pp_gse),
    ("KIRC PFI", "angiogenesis (own axis)", opt_kirc, pp_kirc),
    ("KIRC OS", "angiogenesis (own axis)", opt_kirc_os, pp_kirc_os),
]
print(f"  {'Cancer':<10} {'Axis':<32} {'Transition':>12} {'P':>10}")
for cancer, axis, t, p in summary_rows:
    t_s = f"{t}m" if t is not None else "n/a"
    p_s = f"{p:.4f}" if p is not None else "n/a"
    print(f"  {cancer:<10} {axis:<32} {t_s:>12} {p_s:>10}")


## Save results


In [ ]:
results = {
    "t_breast": T_BREAST,
    "t_luad": T_LUAD,
    "opt_gse2034": opt_gse2034, "pp_gse2034": pp_gse2034,
    "opt_brca_val": opt_brca, "pp_brca_val": pp_brca,
    "opt_gse": opt_gse, "pp_gse": pp_gse,
    "opt_gbm": opt_gbm, "pp_gbm": pp_gbm,
    "opt_kirc": opt_kirc, "pp_kirc": pp_kirc,
    "opt_kirc_os": opt_kirc_os, "pp_kirc_os": pp_kirc_os,
    "breast_phase": breast_phase, "gse_phase": gse_phase,
    "brca_phase": brca_phase, "luad_pfi_phase": luad_pfi_phase,
    "breast_jsd_agg": breast_jsd_agg, "luad_jsd_agg": luad_jsd_agg,
    "brca_jsd_agg": brca_jsd_agg,
    "breast_pw_agg": breast_pw_agg, "luad_pw_agg": luad_pw_agg,
    "breast_fdr_df": breast_fdr_df, "discovery_fdr_df": discovery_fdr_df,
    "bootstrap_results": bs_results,
    "breast_df": breast_df, "gse31210_df": gse31210_df,
    "brca_df": brca_df, "gbm_df": gbm_df, "kirc_df": kirc_df,
}
save_pkl(results, "transition_results")
print("\nNext: 04_pancancer_cox.ipynb")
